In [1]:
# Loome SparkSession-i koos Kafka konnektori ja Delta Lake'i toega.
# spark.jars.packages laeb vajalikud JAR-id Maven Central-ist Sparki käivitumisel.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("metalfab")
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.11")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

Spark 4.1.2


In [2]:
def write_to_postgres(batch_df, batch_id):
    db_url = "jdbc:postgresql://db:5432/praktikum"
    db_properties = {
       "user": "praktikum",
       "password": "praktikum",
       "driver": "org.postgresql.Driver"
        }
    # Kirjutame selle konkreetse partii andmebaasi
    batch_df.write.jdbc(
       url=db_url,
       table="bronze.raw_factory_data",
       mode="append",
       properties=db_properties
       )
    print(f"Batch {batch_id} kirjutatud andmebaasi.")

processed_df = spark.readStream \
    .schema("dept STRING, machine STRING, tag STRING, timestamp TIMESTAMP, topic STRING, value DOUBLE") \
    .json("../data/lake/*/month=06/*/*/*/*")

# 4. Käivita striim
query = processed_df.writeStream \
.foreachBatch(write_to_postgres) \
.option("checkpointLocation", "./checkpoints/postgres_stream") \
.start()


In [7]:
query.status

{'message': 'Getting offsets from FileStreamSource[file:/home/jovyan/data/lake/*/month=06/*/*/*/*]',
 'isDataAvailable': False,
 'isTriggerActive': True}

Batch 1530 kirjutatud andmebaasi.
Batch 1531 kirjutatud andmebaasi.
Batch 1532 kirjutatud andmebaasi.
Batch 1533 kirjutatud andmebaasi.
Batch 1534 kirjutatud andmebaasi.
Batch 1535 kirjutatud andmebaasi.
Batch 1536 kirjutatud andmebaasi.
Batch 1537 kirjutatud andmebaasi.
Batch 1538 kirjutatud andmebaasi.
Batch 1539 kirjutatud andmebaasi.
Batch 1540 kirjutatud andmebaasi.
Batch 1541 kirjutatud andmebaasi.
Batch 1542 kirjutatud andmebaasi.
Batch 1543 kirjutatud andmebaasi.
Batch 1544 kirjutatud andmebaasi.
Batch 1545 kirjutatud andmebaasi.
Batch 1546 kirjutatud andmebaasi.
Batch 1547 kirjutatud andmebaasi.
Batch 1548 kirjutatud andmebaasi.
Batch 1549 kirjutatud andmebaasi.
Batch 1550 kirjutatud andmebaasi.
Batch 1551 kirjutatud andmebaasi.
Batch 1552 kirjutatud andmebaasi.
Batch 1553 kirjutatud andmebaasi.
Batch 1554 kirjutatud andmebaasi.
Batch 1555 kirjutatud andmebaasi.
Batch 1556 kirjutatud andmebaasi.
Batch 1557 kirjutatud andmebaasi.
Batch 1558 kirjutatud andmebaasi.
Batch 1559 kir

In [8]:
#query.stop()